# 01 - Prepare ClinVar Binary Variant Dataset

Purpose: download and preprocess ClinVar GRCh38 VCF data for binary variant classification.

This notebook creates a research-only dataset. It is not a medical diagnosis workflow and must not be used for clinical decision-making.

In [1]:
!git clone https://github.com/faisalAI27/Variant-Risk-Explainer.git /content/variant-risk-explainer

fatal: destination path '/content/variant-risk-explainer' already exists and is not an empty directory.


In [2]:
!pip install -q pandas numpy scikit-learn biopython requests tqdm pyfaidx datasets

In [3]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

import Bio
from datasets import Dataset
from pyfaidx import Fasta

PROJECT_ROOT = Path("/content/variant-risk-explainer")

if not (PROJECT_ROOT / "training" / "utils" / "clinvar_parser.py").exists():
    raise FileNotFoundError(
        f"clinvar_parser.py not found in {PROJECT_ROOT}. "
        "Check your repo was cloned correctly and the file was pushed to GitHub."
    )

if not (PROJECT_ROOT / "training" / "utils" / "sequence_fetcher.py").exists():
    raise FileNotFoundError(
        f"sequence_fetcher.py not found in {PROJECT_ROOT}. "
        "Check the file was pushed to GitHub."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Project root:      {PROJECT_ROOT}")
print(f"✅ Raw data dir:      {RAW_DIR}")
print(f"✅ Processed data dir:{PROCESSED_DIR}")
print(f"✅ clinvar_parser.py  found")
print(f"✅ sequence_fetcher.py found")

✅ Project root:      /content/variant-risk-explainer
✅ Raw data dir:      /content/variant-risk-explainer/data/raw
✅ Processed data dir:/content/variant-risk-explainer/data/processed
✅ clinvar_parser.py  found
✅ sequence_fetcher.py found


In [4]:
CLINVAR_URL = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz"
VCF_PATH = RAW_DIR / "clinvar_grch38.vcf.gz"

def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Using existing file: {destination}")
        return
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))
    with destination.open("wb") as output, tqdm(
        total=total, unit="B", unit_scale=True, desc="Downloading ClinVar GRCh38 VCF"
    ) as progress:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if not chunk:
                continue
            output.write(chunk)
            progress.update(len(chunk))

download_file(CLINVAR_URL, VCF_PATH)
print(f"Downloaded VCF: {VCF_PATH}")
print(f"Size: {VCF_PATH.stat().st_size / (1024 * 1024):.2f} MB")

Using existing file: /content/variant-risk-explainer/data/raw/clinvar_grch38.vcf.gz
Downloaded VCF: /content/variant-risk-explainer/data/raw/clinvar_grch38.vcf.gz
Size: 183.09 MB


In [5]:
from training.utils.clinvar_parser import parse_clinvar_vcf

MAX_RECORDS = None

raw_df = parse_clinvar_vcf(VCF_PATH, max_records=MAX_RECORDS)
print(f"Parsed rows: {len(raw_df):,}")
display(raw_df.head())

Parsed rows: 4,434,969


,CHROM,POS,ID,REF,ALT,INFO,CLNSIG,GENEINFO,CLNHGVS,CLNVC
0,1,66926,3385321,AG,A,ALLELEID=3544463;CLNDISDB=Human_Phenotype_Onto...,Uncertain_significance,OR4F5:79501,NC_000001.11:g.66927del,Deletion
1,1,69134,2205837,A,G,ALLELEID=2193183;CLNDISDB=MedGen:CN169374;CLND...,Likely_benign,OR4F5:79501,NC_000001.11:g.69134A>G,single_nucleotide_variant
2,1,69241,4562067,C,T,ALLELEID=4679177;CLNDISDB=MedGen:CN169374;CLND...,Uncertain_significance,OR4F5:79501,NC_000001.11:g.69241C>T,single_nucleotide_variant
3,1,69308,3925305,A,G,ALLELEID=4039319;CLNDISDB=MedGen:CN169374;CLND...,Uncertain_significance,OR4F5:79501,NC_000001.11:g.69308A>G,single_nucleotide_variant
4,1,69314,3205580,T,G,ALLELEID=3374047;CLNDISDB=MedGen:CN169374;CLND...,Uncertain_significance,OR4F5:79501,NC_000001.11:g.69314T>G,single_nucleotide_variant


In [6]:
from training.utils.clinvar_parser import prepare_binary_variants

clinvar_df = prepare_binary_variants(raw_df)

print(f"Rows after filtering and labeling: {len(clinvar_df):,}")
display(clinvar_df["label_name"].value_counts().rename_axis("label_name").reset_index(name="count"))
display(clinvar_df.head())

Rows after filtering and labeling: 1,856,817


,label_name,count
0,benign_or_likely_benign,1357917
1,pathogenic,498900


,variant_id,CHROM,POS,ID,REF,ALT,variant_type,gene_symbol,GENEINFO,CLNSIG,CLNHGVS,CLNVC,label,label_name
0,GRCh38-1-69134-A-G,1,69134,2205837,A,G,SNV,OR4F5,OR4F5:79501,Likely_benign,NC_000001.11:g.69134A>G,single_nucleotide_variant,0,benign_or_likely_benign
1,GRCh38-1-809284-T-TGGTCAATCA,1,809284,3892489,T,TGGTCAATCA,INDEL,LINC01409,LINC01409:105378580,Benign,NC_000001.11:g.809289_809290insATCAGGTCA,Insertion,0,benign_or_likely_benign
2,GRCh38-1-924518-G-C,1,924518,3388928,G,C,SNV,SAMD11,SAMD11:148398|LOC107985728:107985728,Likely_benign,NC_000001.11:g.924518G>C,single_nucleotide_variant,0,benign_or_likely_benign
3,GRCh38-1-925956-C-T,1,925956,1543320,C,T,SNV,SAMD11,SAMD11:148398,Likely_benign,NC_000001.11:g.925956C>T,single_nucleotide_variant,0,benign_or_likely_benign
4,GRCh38-1-925969-C-T,1,925969,1648427,C,T,SNV,SAMD11,SAMD11:148398,Likely_benign,NC_000001.11:g.925969C>T,single_nucleotide_variant,0,benign_or_likely_benign


In [7]:
assert clinvar_df["variant_id"].str.startswith("GRCh38-").all()
duplicate_variant_ids = int(clinvar_df["variant_id"].duplicated().sum())
print(f"Duplicate variant_id rows: {duplicate_variant_ids:,}")
display(clinvar_df[["variant_id", "CHROM", "POS", "REF", "ALT", "label", "label_name"]].head())

Duplicate variant_id rows: 0


,variant_id,CHROM,POS,REF,ALT,label,label_name
0,GRCh38-1-69134-A-G,1,69134,A,G,0,benign_or_likely_benign
1,GRCh38-1-809284-T-TGGTCAATCA,1,809284,T,TGGTCAATCA,0,benign_or_likely_benign
2,GRCh38-1-924518-G-C,1,924518,G,C,0,benign_or_likely_benign
3,GRCh38-1-925956-C-T,1,925956,C,T,0,benign_or_likely_benign
4,GRCh38-1-925969-C-T,1,925969,C,T,0,benign_or_likely_benign


In [8]:

if clinvar_df["label"].nunique() != 2:
    raise ValueError("Expected both binary classes before stratified splitting.")

train_df, temp_df = train_test_split(
    clinvar_df, test_size=0.30, stratify=clinvar_df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42
)

print(f"Train rows: {len(train_df):,} ({len(train_df) / len(clinvar_df):.1%})")
print(f"Val rows:   {len(val_df):,} ({len(val_df) / len(clinvar_df):.1%})")
print(f"Test rows:  {len(test_df):,} ({len(test_df) / len(clinvar_df):.1%})")

Train rows: 1,299,771 (70.0%)
Val rows:   278,523 (15.0%)
Test rows:  278,523 (15.0%)


In [9]:
clinvar_df.to_csv(PROCESSED_DIR / "clinvar_binary_variants.csv", index=False)
train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "val.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

for name in ["clinvar_binary_variants.csv", "train.csv", "val.csv", "test.csv"]:
    p = PROCESSED_DIR / name
    print(f"✅ {name} — {p.stat().st_size / (1024*1024):.2f} MB")

✅ clinvar_binary_variants.csv — 285.55 MB
✅ train.csv — 199.88 MB
✅ val.csv — 42.84 MB
✅ test.csv — 42.83 MB


In [10]:
from training.utils.sequence_fetcher import add_sequences_to_dataframe, build_sequence_fetcher

FLANK_SIZE = 512
MIN_SEQ_LEN = 200
SAMPLE_MODE = True       # Set False for full dataset (slow — 100k+ UCSC API calls)
SAMPLE_SIZE = 1000
SEQUENCE_FETCH_MODE = "ucsc"
LOCAL_FASTA_PATH = ""
UCSC_SLEEP_SECONDS = 0.15
UCSC_MAX_RETRIES = 3
UCSC_CACHE_PATH = PROCESSED_DIR / f"ucsc_sequence_cache_flank{FLANK_SIZE}.json"

print(f"FLANK_SIZE:            {FLANK_SIZE}")
print(f"MIN_SEQ_LEN:           {MIN_SEQ_LEN}")
print(f"SAMPLE_MODE:           {SAMPLE_MODE}")
print(f"SEQUENCE_FETCH_MODE:   {SEQUENCE_FETCH_MODE}")
print(f"UCSC cache path:       {UCSC_CACHE_PATH.relative_to(PROJECT_ROOT)}")

FLANK_SIZE:            512
MIN_SEQ_LEN:           200
SAMPLE_MODE:           True
SEQUENCE_FETCH_MODE:   ucsc
UCSC cache path:       data/processed/ucsc_sequence_cache_flank512.json


In [11]:
def combine_splits_for_sequence_fetching(train_frame, val_frame, test_frame):
    frames = []
    for split_name, frame in [("train", train_frame), ("val", val_frame), ("test", test_frame)]:
        split_frame = frame.copy()
        split_frame["_split"] = split_name
        frames.append(split_frame)
    return pd.concat(frames, ignore_index=True)

combined_for_sequences = combine_splits_for_sequence_fetching(train_df, val_df, test_df)

if SAMPLE_MODE:
    sample_n = min(SAMPLE_SIZE, len(combined_for_sequences))
    stratify_values = (
        combined_for_sequences["_split"].astype(str)
        + "_"
        + combined_for_sequences["label"].astype(str)
    )
    if sample_n < len(combined_for_sequences) and stratify_values.value_counts().min() >= 2:
        _, sequence_source_df = train_test_split(
            combined_for_sequences, test_size=sample_n, stratify=stratify_values, random_state=42
        )
    else:
        sequence_source_df = combined_for_sequences.sample(n=sample_n, random_state=42)
else:
    sequence_source_df = combined_for_sequences.copy()

sequence_source_splits = {}
for split_name in ["train", "val", "test"]:
    split_frame = sequence_source_df.loc[sequence_source_df["_split"] == split_name].copy()
    sequence_source_splits[split_name] = split_frame.drop(columns=["_split"])
    print(f"{split_name}: {len(sequence_source_splits[split_name]):,} variants selected")

train: 700 variants selected
val: 150 variants selected
test: 150 variants selected


In [12]:
fetcher = build_sequence_fetcher(
    mode=SEQUENCE_FETCH_MODE,
    fasta_path=LOCAL_FASTA_PATH or None,
    cache_path=UCSC_CACHE_PATH if SEQUENCE_FETCH_MODE == "ucsc" else None,
    sleep_seconds=UCSC_SLEEP_SECONDS,
    max_retries=UCSC_MAX_RETRIES,
)

sequence_outputs = {}
sequence_stats = []

for split_name, split_frame in sequence_source_splits.items():
    with_sequences, stats = add_sequences_to_dataframe(
        split_frame,
        fetcher=fetcher,
        flank_size=FLANK_SIZE,
        min_seq_len=MIN_SEQ_LEN,
        progress_desc=f"{split_name}: fetching sequences",
    )
    sequence_outputs[split_name] = with_sequences
    stats["split"] = split_name
    sequence_stats.append(stats)

if hasattr(fetcher, "save_cache"):
    fetcher.save_cache()

sequence_stats_df = pd.DataFrame(sequence_stats)
display(sequence_stats_df[["split", "input_rows", "output_rows", "failed_fetches", "too_short", "dropped_rows"]])

train: fetching sequences:   0%|          | 0/700 [00:00<?, ?it/s]

val: fetching sequences:   0%|          | 0/150 [00:00<?, ?it/s]

test: fetching sequences:   0%|          | 0/150 [00:00<?, ?it/s]

,split,input_rows,output_rows,failed_fetches,too_short,dropped_rows
0,train,700,697,3,0,3
1,val,150,150,0,0,0
2,test,150,150,0,0,0


In [13]:
for split_name in ["train", "val", "test"]:
    out_path = PROCESSED_DIR / f"{split_name}_with_sequences.csv"
    sequence_outputs[split_name].to_csv(out_path, index=False)
    print(f"✅ {out_path.name} — {out_path.stat().st_size / (1024*1024):.2f} MB")

# Download all 3 files immediately
from google.colab import files
for split_name in ["train", "val", "test"]:
    files.download(str(PROCESSED_DIR / f"{split_name}_with_sequences.csv"))

✅ train_with_sequences.csv — 0.79 MB
✅ val_with_sequences.csv — 0.17 MB
✅ test_with_sequences.csv — 0.17 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>